# Base modelling — granularity isolation (2005–2017), EXTENDED macro set

**Purpose:** re-run of the country-level **extended** macro-only model on the *same window as the bank-level extended baseline (NB11b)* — 2005–2017, testing 2010/2011/2012 — so the only difference vs the NB11b macro config is **granularity** (country-year vs bank-year + LR country dummies), with the training-window confound removed.

Pairs with `12a_macro_base_isolate_gran.ipynb` (baseline set). Together they give the granularity-only version of Figure 5 (baseline + extended). Requires `macro_ind_robust_me.parquet` regenerated by `script_merge_and_engineer.ipynb` **after** the one-sided HP fix (asserted in the load cell). Output: `results_12b_isolate_gran/`.

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, confusion_matrix,
    accuracy_score, f1_score, recall_score, precision_score,
    average_precision_score, brier_score_loss
)
# ../ navigates up from notebooks/ to the repo root
df = pd.read_parquet("../data/individual_merges/parquet_saves/macro_ind_robust_me.parquet")

# --- Granularity-isolation window (2005-2017), EXTENDED macro set ------------
# Sister run to 12a_macro_base_isolate_gran, for the EXTENDED indicator set.
# Re-runs the country-level extended macro-only model on the SAME window as the
# bank-level extended baseline (NB11b) -> 2005-2017, testing 2010/2011/2012 - so
# the only difference vs the NB11b macro config is GRANULARITY (country-year vs
# bank-year + LR country dummies), with the training-window confound removed
# (the original 12b trains from 2001).
#
# IMPORTANT: this reads macro_ind_robust_me.parquet, which MUST have been
# regenerated by script_merge_and_engineer.ipynb AFTER the one-sided HP fix in
# NB07. Otherwise credit_gap here is the stale two-sided version and is NOT
# comparable to NB11b (one-sided). The assert below guards against that: it
# checks Spain's 2007 credit_gap is ~+3.3 (one-sided), not ~+16 (two-sided).
_esp07 = df[(df["country_iso"] == "ESP") & (df["year"] == 2007)]["credit_gap"]
assert _esp07.empty or _esp07.iloc[0] < 8, (
    f"credit_gap looks TWO-SIDED (ESP 2007 = {_esp07.iloc[0]:.2f}); "
    "re-run script_merge_and_engineer.ipynb after the NB07 one-sided fix first."
)
df = df[df["year"].between(2005, 2017)].reset_index(drop=True)
print("Isolation window:", int(df["year"].min()), "-", int(df["year"].max()), "| rows:", len(df))
print(df.columns.tolist())

In [ ]:
country_col = 'country_iso'
target = 'crisis'
id_cols = [country_col, 'year', target]
feat_cols = [col for col in df.columns if col not in id_cols]

# Collinearity identification

## VIF

In [ ]:
def compute_vif(df, cols):
    tmp = df[cols].dropna() #selects only columns you want to check, and drops na since it cannot handle missing values
    vif_data = pd.DataFrame({
        'feature': cols,
        'VIF': [variance_inflation_factor(tmp.values, i) for i in range(len(cols))] #takes full feature matrix and index of column to check, then returns VIF score
    })
    return vif_data.sort_values('VIF', ascending=False)

In [ ]:
print("\n=== Macro ===")
print(compute_vif(df, feat_cols))

BIS_REER > 10, thus is dropped

In [ ]:
feat_cols.remove("BIS_REER")
feat_cols.remove('FM.AST.NFRG.CN') #dropped as it is only used to compute a derived variable

# Expanding window split

In [ ]:
def expanding_window_split(df, min_train_years):
    years = sorted(df['year'].unique())
    splits = []

    for i, test_year in enumerate(years):
        if i < min_train_years:
            continue
        train = df[df['year'] < test_year]
        test = df[df['year'] == test_year]
        splits.append((train, test))
    return splits

In [ ]:
splits = expanding_window_split(df, min_train_years=5)
for train, test in splits[:3]:
    print(f"Train: {train['year'].min()}-{train['year'].max()}, Test: {test['year'].values[0]}")

# Results dataframe

In [ ]:
results = pd.DataFrame(columns = [
    'model', 'variant', 'dataset', 'test_year',
    'auroc', 'auprc', 'brier', 'best_threshold',
    'tp', 'fp', 'fn', 'tn',
    'f1', 'recall', 'precision',
    'miss_rate', 'false_alarm_rate', 'accuracy',
    'loss_07', 'loss_08', 'loss_09'
])

# Run_model helper

In [ ]:
def run_model(model, X_train, y_train, X_test, y_test, metadata, sample_weight=None, return_proba=False):
    # sample_weight is only used by MLP -- LR/RF/XGBoost handle imbalance via class_weight/scale_pos_weight
    fit_kwargs = {'sample_weight': sample_weight} if sample_weight is not None else {}
    model.fit(X_train, y_train, **fit_kwargs)
    y_proba = model.predict_proba(X_test)[:, 1]

    # Find best threshold by maximising F1
    # np.where evaluates both branches eagerly, so the divide runs even where denom=0.
    # errstate suppresses the warning; np.where then correctly replaces those positions with 0.0.
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
    denom     = precisions + recalls
    with np.errstate(invalid='ignore', divide='ignore'):
        f1_scores = np.where(denom > 0, 2 * precisions * recalls / denom, 0.0)
    best_threshold = thresholds[np.argmax(f1_scores[:-1])]
    y_pred = (y_proba >= best_threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    auroc            = roc_auc_score(y_test, y_proba)
    auprc            = average_precision_score(y_test, y_proba)
    brier            = brier_score_loss(y_test, y_proba)
    f1               = f1_score(y_test, y_pred, zero_division=0)
    recall           = recall_score(y_test, y_pred, zero_division=0)
    precision        = precision_score(y_test, y_pred, zero_division=0)
    miss_rate        = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    false_alarm_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    accuracy         = accuracy_score(y_test, y_pred)

    loss_07 = 0.7 * miss_rate + 0.3 * false_alarm_rate
    loss_08 = 0.8 * miss_rate + 0.2 * false_alarm_rate
    loss_09 = 0.9 * miss_rate + 0.1 * false_alarm_rate

    result = {
        **metadata,
        'auroc':            auroc,
        'auprc':            auprc,
        'brier':            brier,
        'best_threshold':   best_threshold,
        'tp':               int(tp),
        'fp':               int(fp),
        'fn':               int(fn),
        'tn':               int(tn),
        'f1':               f1,
        'recall':           recall,
        'precision':        precision,
        'miss_rate':        miss_rate,
        'false_alarm_rate': false_alarm_rate,
        'accuracy':         accuracy,
        'loss_07':          loss_07,
        'loss_08':          loss_08,
        'loss_09':          loss_09,
    }

    # Only include raw probabilities when requested -- used by the main training loop
    # to populate proba_store for DeLong tests. Diagnostic loops (H3, no-dummies)
    # don't request them so their DataFrames stay clean.
    if return_proba:
        result['_y_proba'] = list(y_proba)
        result['_y_true']  = list(y_test)

    return result

# Training loop

In [ ]:
import optuna
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
import lightgbm as lgb

# Suppress per-trial Optuna output — only warnings and errors will print
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
def get_objective(model_name, X_tr_raw, y_tr, X_vl_raw, y_vl, n_neg_ratio):
    """
    Returns an Optuna objective function for the given model type.

    Imputation (and scaling for LR/MLP) are done inside each trial, fitted on inner_train only.
    This mirrors the full pipeline and prevents any leakage from inner_val into inner_train.
    Objective: maximise AUROC on the inner validation set.

    Parameters
    ----------
    X_tr_raw, X_vl_raw : pd.DataFrame — raw (pre-imputed) feature matrices
    y_tr, y_vl         : pd.Series   — crisis labels
    n_neg_ratio        : float        — n_neg / n_pos in the full training window (for XGBoost/MLP)
    """
    def objective(trial):
        # Fit imputer on inner_train only — same leakage rule as the outer loop
        imputer = SimpleImputer(strategy='median')
        X_tr = imputer.fit_transform(X_tr_raw)
        X_vl = imputer.transform(X_vl_raw)

        if model_name == 'logistic_regression':
            C = trial.suggest_float('C', 1e-4, 10.0, log=True)
            # Scale inside trial — fit on inner_train only
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_tr)
            X_vl = scaler.transform(X_vl)
            # L2 (Ridge): shrinks all coefficients toward zero but keeps all features.
            # Chosen over L1 because we want a consistent feature set across model classes
            # for cross-model SHAP comparison — L1 would zero out different features each window.
            # lbfgs is the recommended solver for L2; faster and more stable than saga for small-medium data.
            model = LogisticRegression(
                solver='lbfgs', C=C,
                max_iter=1000, class_weight='balanced', random_state=42
            )

        elif model_name == 'random_forest':
            model = RandomForestClassifier(
                n_estimators=trial.suggest_int('n_estimators', 50, 500),
                max_depth=trial.suggest_int('max_depth', 3, 20),
                min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 20),
                max_features=trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5]),
                class_weight='balanced', random_state=42,
            )

        elif model_name == 'xgboost':
            model = XGBClassifier(
                n_estimators=trial.suggest_int('n_estimators', 50, 500),
                max_depth=trial.suggest_int('max_depth', 2, 10),
                learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                subsample=trial.suggest_float('subsample', 0.5, 1.0),
                min_child_weight=trial.suggest_int('min_child_weight', 1, 10),
                scale_pos_weight=n_neg_ratio,
                eval_metric='logloss', random_state=42
            )

        elif model_name == 'mlp':
            # MLP needs scaled inputs — gradient descent is sensitive to feature magnitude
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_tr)
            X_vl = scaler.transform(X_vl)
            # suggest_categorical only accepts primitives — encode arch as string, decode to tuple
            arch = trial.suggest_categorical('hidden_layer_sizes', ['32', '64', '128', '64_32', '128_64'])
            # single-layer options (32/64/128): small to large capacity for limited panel data
            # two-layer funnels (64_32/128_64): standard progressive narrowing for tabular finance
            hidden_layer_sizes = {'32': (32,), '64': (64,), '128': (128,), '64_32': (64, 32), '128_64': (128, 64)}[arch]
            model = MLPClassifier(
                hidden_layer_sizes=hidden_layer_sizes,
                learning_rate_init=trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
                alpha=trial.suggest_float('alpha', 1e-5, 1e-2, log=True),
                max_iter=5000,
                tol=1e-3,  # relaxed from default 1e-4 — suppresses warnings on flat loss landscapes without affecting AUROC
                random_state=42
            )

        elif model_name == 'decision_tree':
            model = DecisionTreeClassifier(
            max_depth=trial.suggest_int('max_depth', 2, 15),
            min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 30),
            class_weight='balanced', random_state=42
            )
    
        elif model_name == 'lightgbm':
            # Ensure consistent feature names for fit and predict
            # SimpleImputer may return a DataFrame in newer sklearn versions
            if not isinstance(X_tr, pd.DataFrame):
                X_tr = pd.DataFrame(X_tr, columns=X_tr_raw.columns)
                X_vl = pd.DataFrame(X_vl, columns=X_vl_raw.columns)
            model = lgb.LGBMClassifier(
            n_estimators=trial.suggest_int('n_estimators', 50, 500),
            max_depth=trial.suggest_int('max_depth', 2, 10),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            num_leaves=trial.suggest_int('num_leaves', 15, 63),
            subsample=trial.suggest_float('subsample', 0.5, 1.0),
            min_child_samples=trial.suggest_int('min_child_samples', 5, 30),
            scale_pos_weight=n_neg_ratio,
            random_state=42, verbose=-1
        )

        # MLP has no class_weight parameter — pass sample_weight to handle class imbalance
        # All other models already handle imbalance via class_weight / scale_pos_weight
        if model_name == 'mlp':
            sw = np.where(y_tr == 1, n_neg_ratio, 1.0)
            model.fit(X_tr, y_tr, sample_weight=sw)
        else:
            model.fit(X_tr, y_tr)
        y_proba = model.predict_proba(X_vl)[:, 1]
        return roc_auc_score(y_vl, y_proba)
        
    return objective

## What's not tuned (fixed by design)


Each model has parameters that are **held fixed** — they are design choices that apply across all windows and variants, not tuning targets.

| Model | Fixed parameter | Value | Why fixed |
|-------|----------------|-------|-----------|
| All | `random_state` | `42` | Reproducibility across runs |
| LR | `solver` | `lbfgs` | Best for L2 penalty; faster and more stable than `saga` on small-medium panel data |
| LR | `penalty` | `l2` | Chosen for SHAP cross-model comparability — L1 would zero out different features per window, making SHAP values incomparable |
| LR | `max_iter` | `1000` | Sufficient for convergence; avoids `ConvergenceWarning` |
| LR, RF, DT | `class_weight` | `'balanced'` | Imbalance correction — a modelling principle, not a tuning target. Sklearn computes inverse-frequency weights automatically |
| XGBoost, LightGBM | `scale_pos_weight` | `n_neg / n_pos` | Same rationale as `class_weight`; XGB/LGBM use this scalar instead. Computed per window so it tracks the actual class ratio in each training set |
| XGBoost | `eval_metric` | `'logloss'` | Standard for binary classification with `predict_proba` |
| MLP | `activation` | `'relu'` (default) | Near-universal default for tabular data; no theoretical reason to tune it here |
| MLP | `max_iter` | `5000` | Generous budget to allow convergence on small crisis datasets without inflating wall time |
| MLP | `tol` | `1e-3` | Relaxed from sklearn default (`1e-4`) — suppresses warnings on flat loss landscapes in short inner windows; does not affect AUROC |
| MLP | `sample_weight` | `n_neg / n_pos` | MLP has no `class_weight` param — sample weights replicate the same imbalance correction used by the other models |
| DT | `criterion` | `'gini'` (default) | Gini and entropy give near-identical splits in practice; not worth a tuning dimension |
| LightGBM | `verbose` | `-1` | Suppress per-tree console output only — no effect on training |

In [ ]:
MODEL_NAMES = ['logistic_regression', 'random_forest', 'xgboost', 'mlp', 'decision_tree', 'lightgbm']
N_TRIALS = 50  # Optuna trials per model per window

# Two variants:
#   macro_standalone — full training window, no exclusions
#   rt1              — exclude crisis year (t) and post-crisis year (t+1) from training, per country.
#                      Non-crisis countries keep all their data even in crisis calendar years.
#                      Rationale: crisis/post-crisis bank profiles are "disorderly adjustments"
#                      not informative about pre-crisis vulnerability (Bussiere & Fratzscher 2006).
#                      With duration labeling, RT1 targets the ONSET year + t+1 only (not the full
#                      duration), because the sustained crisis period is itself informative.
VARIANTS = [('macro_standalone', False), ('rt1', True)]

all_results     = []
best_params_log = {}  # keyed by (variant, dataset, test_year, model)
proba_store     = {}  # (model, dataset, test_year, variant) → (y_true_list, y_proba_list)
fi_store        = {}  # (model, dataset, test_year, variant) → {feature: importance} (None for MLP)


for variant_name, use_rt1 in VARIANTS:
        dataset_name = 'macro'
        cols = feat_cols

        print(f"\n=== Variant: {variant_name} | Dataset: {dataset_name} ===")

        for train_full, test in splits:
            test_year = test['year'].values[0]
            y_test    = test[target]

            if y_test.sum() == 0:
                print(f"  test={test_year} | SKIPPED — no crisis obs in test year")
                continue

            # RT1: build the set of (country, year) pairs to exclude — onset year + t+1.
            # We identify onset years as country-years where crisis first switches to 1
            # (i.e. crisis=1 but crisis=0 in the preceding year for that country).
            # Only those rows (and their t+1) are dropped; the rest of the crisis duration stays.
            if use_rt1:
                df_sorted   = train_full.sort_values([country_col, 'year'])
                prev_crisis = df_sorted.groupby(country_col)[target].shift(1).fillna(0)
                onset_mask  = (df_sorted[target] == 1) & (prev_crisis == 0)
                onset_rows  = df_sorted[onset_mask][[country_col, 'year']]
                exclude_set = (
                    set(zip(onset_rows[country_col], onset_rows['year'])) |
                    set(zip(onset_rows[country_col], onset_rows['year'] + 1))
                )
                train = train_full[~train_full.apply(
                    lambda r: (r[country_col], r['year']) in exclude_set, axis=1
                )]
                if train[target].sum() == 0:
                    print(f"  test={test_year} | RT1 SKIPPED — no crisis obs after onset filter")
                    continue
            else:
                train = train_full

            y_train     = train[target]
            n_pos       = (y_train == 1).sum()
            n_neg       = (y_train == 0).sum()
            n_neg_ratio = n_neg / n_pos if n_pos > 0 else 1.0

            # Inner split: last 2 years of the (possibly filtered) training window
            train_years     = sorted(train['year'].unique())
            inner_val_years = set(train_years[-2:])
            inner_train     = train[~train['year'].isin(inner_val_years)]
            inner_val       = train[train['year'].isin(inner_val_years)]
            y_inner_train   = inner_train[target]
            y_inner_val     = inner_val[target]

            can_tune = y_inner_val.sum() > 0 and y_inner_train.sum() > 0
            print(f"  test={test_year} | val={sorted(inner_val_years)} | crisis_in_val={can_tune}")

            for model_name in MODEL_NAMES:

                # --- Inner loop: Optuna tunes hyperparameters on inner_train/inner_val ---
                if can_tune:
                    objective = get_objective(
                        model_name,
                        inner_train[cols], y_inner_train,
                        inner_val[cols],   y_inner_val,
                        n_neg_ratio
                    )
                    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
                    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
                    best_params = study.best_params
                else:
                    best_params = {}

                best_params_log[(variant_name, dataset_name, test_year, model_name)] = best_params

                # --- Outer loop: refit on full (filtered) training window ---
                imputer     = SimpleImputer(strategy='median')
                X_train_imp = imputer.fit_transform(train[cols])
                X_test_imp  = imputer.transform(test[cols])

                if model_name == 'logistic_regression':
                    scaler        = StandardScaler()
                    X_train_final = scaler.fit_transform(X_train_imp)
                    X_test_final  = scaler.transform(X_test_imp)
                    model = LogisticRegression(
                        solver='lbfgs',
                        C=best_params.get('C', 1.0),
                        max_iter=1000, class_weight='balanced', random_state=42
                    )

                elif model_name == 'random_forest':
                    X_train_final = X_train_imp
                    X_test_final  = X_test_imp
                    model = RandomForestClassifier(
                        n_estimators=best_params.get('n_estimators', 100),
                        max_depth=best_params.get('max_depth', None),
                        min_samples_leaf=best_params.get('min_samples_leaf', 1),
                        max_features=best_params.get('max_features', 'sqrt'),
                        class_weight='balanced', random_state=42
                    )

                elif model_name == 'xgboost':
                    X_train_final = X_train_imp
                    X_test_final  = X_test_imp
                    model = XGBClassifier(
                        n_estimators=best_params.get('n_estimators', 100),
                        max_depth=best_params.get('max_depth', 6),
                        learning_rate=best_params.get('learning_rate', 0.1),
                        subsample=best_params.get('subsample', 1.0),
                        min_child_weight=best_params.get('min_child_weight', 1),
                        scale_pos_weight=n_neg_ratio,
                        eval_metric='logloss', random_state=42
                    )

                elif model_name == 'mlp':
                    scaler        = StandardScaler()
                    X_train_final = scaler.fit_transform(X_train_imp)
                    X_test_final  = scaler.transform(X_test_imp)
                    arch = best_params.get('hidden_layer_sizes', '128')
                    hidden_layer_sizes = {'32': (32,), '64': (64,), '128': (128,), '64_32': (64, 32), '128_64': (128, 64)}.get(arch, (64,))
                    model = MLPClassifier(
                        hidden_layer_sizes=hidden_layer_sizes,
                        learning_rate_init=best_params.get('learning_rate_init', 1e-3),
                        alpha=best_params.get('alpha', 1e-4),
                        max_iter=5000,
                        tol=1e-3,  # relaxed from default 1e-4 — suppresses warnings on flat loss landscapes without affecting AUROC
                        random_state=42
                    )

                elif model_name == 'decision_tree':
                    X_train_final = X_train_imp
                    X_test_final  = X_test_imp
                    model = DecisionTreeClassifier(
                        max_depth=best_params.get('max_depth', 5),
                        min_samples_leaf=best_params.get('min_samples_leaf', 5),
                        class_weight='balanced', random_state=42
                    )

                elif model_name == 'lightgbm':
                    # Wrap as DataFrame: ensures consistent feature names for fit and predict
                    X_train_final = pd.DataFrame(X_train_imp, columns=cols)
                    X_test_final  = pd.DataFrame(X_test_imp,  columns=cols)
                    model = lgb.LGBMClassifier(
                        n_estimators=best_params.get('n_estimators', 100),
                        max_depth=best_params.get('max_depth', 6),
                        learning_rate=best_params.get('learning_rate', 0.1),
                        num_leaves=best_params.get('num_leaves', 31),
                        subsample=best_params.get('subsample', 1.0),
                        min_child_samples=best_params.get('min_child_samples', 20),
                        scale_pos_weight=n_neg_ratio,
                        random_state=42, verbose=-1
                    )

                sw = np.where(y_train == 1, n_neg_ratio, 1.0) if model_name == 'mlp' else None
                result = run_model(
                    model, X_train_final, y_train, X_test_final, y_test,
                    metadata={
                        'model':     model_name,
                        'variant':   variant_name,
                        'dataset':   dataset_name,
                        'test_year': test_year,
                    },
                    sample_weight=sw,
                    return_proba=True,
                )
                # Store raw probabilities for DeLong test, then remove from result
                # so they don't appear as columns in the results DataFrame
                proba_store[(model_name, dataset_name, test_year, variant_name)] = (
                    result.pop('_y_true'), result.pop('_y_proba')
                )
                # Tree models expose feature_importances_, LR exposes coef_
                # MLP has no native importance (None here; use SHAP in NB13 if needed)
                if hasattr(model, 'feature_importances_'):
                    fi_store[(model_name, dataset_name, test_year, variant_name)] = dict(zip(cols, model.feature_importances_))
                elif hasattr(model, 'coef_'):
                    fi_store[(model_name, dataset_name, test_year, variant_name)] = dict(zip(cols, model.coef_[0]))
                else:
                    fi_store[(model_name, dataset_name, test_year, variant_name)] = None
                all_results.append(result)

if not all_results:
    print("\nWARNING: no results collected — every test year had zero crisis observations.")
    print(f"Crisis counts by year:\n{df.groupby('year')[target].sum()}")
else:
    results = pd.DataFrame(all_results)
    print(f"\nCompleted: {len(results)} model-window evaluations")
    print(f"\nMean AUROC by model (Macro only):")
    print(
        results[results['variant'] == 'macro_standalone']
        .groupby(['model', 'dataset'])['auroc'].mean().round(3).unstack()
    )
    print(f"\nMean AUROC by model and dataset (rt1):")
    print(
        results[results['variant'] == 'rt1']
        .groupby(['model', 'dataset'])['auroc'].mean().round(3).unstack()
    )

# LR sanity check

In [ ]:
# === LR Sanity Check ===
# Two concerns flagged during initial run:
# (1) Suspiciously high AUROC (~0.997) for LR on micro — check whether this is year-specific
#     or consistent across the evaluation window (consistent = country dummies encoding crisis membership)
# (2) L2 regularisation shrinks all coefficients toward zero — verify that financial ratios
#     carry meaningful weight and the model is not relying almost entirely on country dummies

# (1) AUROC by year for LR
lr_results = results[(results['model'] == 'logistic_regression') & (results['variant'] == 'macro_standalone')]
print("=== LR AUROC by test_year and dataset (macro_standalone) ===")
print(lr_results.pivot_table(index='test_year', columns='dataset', values='auroc').round(3).to_string())

# (2) Refit LR on the last available split and inspect coefficients
last_train, last_test = splits[-1]
cols_inspect = feat_cols

imputer_inspect        = SimpleImputer(strategy='median')
X_train_imp_inspect    = imputer_inspect.fit_transform(last_train[cols_inspect])
scaler_inspect         = StandardScaler()
X_train_scaled_inspect = scaler_inspect.fit_transform(X_train_imp_inspect)

# best_params_log key is now (variant, dataset, test_year, model)
last_year  = last_test['year'].values[0]
last_best  = best_params_log.get(('macro_standalone', 'macro', last_year, 'logistic_regression'), {})
C_inspect  = last_best.get('C', 1.0)

lr_inspect = LogisticRegression(
    solver='lbfgs', C=C_inspect,
    max_iter=1000, class_weight='balanced', random_state=42
)
lr_inspect.fit(X_train_scaled_inspect, last_train[target])

coef_df = pd.DataFrame({
    'feature': cols_inspect,
    'coef':    lr_inspect.coef_[0]
}).sort_values('coef', key=abs, ascending=False)

n_nonzero = (coef_df['coef'] != 0).sum()
print(f"\n=== LR coefficients — last window (test={last_year}), macro, C={C_inspect:.4f} ===")
print(f"Non-zero: {n_nonzero} / {len(coef_df)} features")
print(coef_df.head(20).to_string(index=False))

## Save all results

In [ ]:
import os
import pandas as pd

# Separate output folder — keeps the original NB12b (extended-window) results intact.
out_dir = "../outputs/results/12b_gran"
os.makedirs(out_dir, exist_ok=True)

with pd.ExcelWriter(f"{out_dir}/results_12b_isolate_gran_final.xlsx", engine="openpyxl") as writer:

    # --- Main results ---
    results.to_excel(writer, sheet_name="all_results", index=False)

    results[results["variant"] == "macro_standalone"].pivot_table(
        index="model", columns="dataset", values="auroc", aggfunc="mean"
    ).round(3).to_excel(writer, sheet_name="auroc_pivot_baseline")

    results[results["variant"] == "macro_standalone"].groupby(["model", "dataset"])[
        ["auroc", "f1", "recall", "precision", "miss_rate", "false_alarm_rate",
  "loss_08"]
    ].mean().round(3).to_excel(writer, sheet_name="mean_metrics_baseline")

    results.groupby(["variant", "model", "dataset"])["auroc"].mean().round(3)\
        .unstack("dataset").to_excel(writer, sheet_name="macro_standalone_vs_rt1")

print(f"Saved to {out_dir}/results_12b_isolate_gran_final.xlsx")

# Save proba_store — raw (y_true, y_proba) per model/window; needed for PR/ROC curves in NB13
with open(f"{out_dir}/proba_store_final.pkl", "wb") as f:
    pickle.dump(proba_store, f)
print(f"[SAVED] {out_dir}/proba_store_final.pkl")

# Save best_params_log — hyperparameter choices per model/window; for methodology table in NB13
with open(f"{out_dir}/best_params_log_final.pkl", "wb") as f:
    pickle.dump(best_params_log, f)
print(f"[SAVED] {out_dir}/best_params_log_final.pkl")

# Convert fi_store to long-format DataFrame and save; needed for feature importance plots in NB13
fi_rows = []
for (mdl, ds, yr, var), fi in fi_store.items():
    if fi is None:
        continue
    for feat, imp in fi.items():
        fi_rows.append({"model": mdl, "dataset": ds, "test_year": yr,
                        "variant": var, "feature": feat, "importance": imp})
fi_df = pd.DataFrame(fi_rows)
fi_df.to_parquet(f"{out_dir}/fi_store_final.parquet")
print(f"[SAVED] {out_dir}/fi_store_final.parquet")

In [ ]:
import pandas as pd
df = pd.read_parquet("../data/individual_merges/parquet_saves/macro_ind_robust_me.parquet")
print(df[df["country_iso"] == "SVK"][["year", "crisis"]].sort_values("year"))